# 08 Time Alignment and Lead Time Framework

## Purpose

This notebook defines the timing framework used to compare AI weather forecasts with Polymarket prices.

The key principle is that a forecast should be compared with the market price observed at the forecast information time, not at the forecast valid time or after the event has become easier to predict. This avoids look ahead bias.

For example, if a forecast is issued at 12:00 UTC on 29 May and is valid at 12:00 UTC on 30 May, the relevant market price should be sampled at or shortly after 12:00 UTC on 29 May. The realised temperature is used only after the event for scoring.

This notebook therefore formalises:

- forecast run time;
- forecast issue time;
- forecast valid time;
- target local date;
- local settlement window;
- lead time;
- market price timestamp;
- realised settlement value.

## 1. Imports and paths

In [3]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from io import StringIO
import json
import ast

import requests
import pandas as pd
import numpy as np

RAW_DIR = Path("../data/raw/time_alignment")
PROCESSED_DIR = Path("../data/processed/time_alignment")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

print("Notebook run time UTC:", datetime.now(timezone.utc).isoformat())
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

Notebook run time UTC: 2026-06-16T23:51:24.977649+00:00
Raw directory: ../data/raw/time_alignment
Processed directory: ../data/processed/time_alignment


## 2. Contract and settlement metadata

The first example uses the Hong Kong market selected in the settlement-source notebook. The official realised value is retrieved from the HKO settlement table if available.

In [5]:
contract_metadata = {
    "city": "Hong Kong",
    "local_timezone": "Asia/Hong_Kong",
    "contract_slug": "highest-temperature-in-hong-kong-on-may-30-2026",
    "polymarket_event_url": "https://polymarket.com/event/highest-temperature-in-hong-kong-on-may-30-2026",
    "settlement_date_local": "2026-05-30",
    "settlement_source": "Hong Kong Observatory",
    "settlement_variable": "Daily maximum temperature",
    "unit": "deg C",
}

settlement_path = Path("../data/processed/hko/hko_official_settlement_temperature.csv")

if settlement_path.exists():
    settlement_df = pd.read_csv(settlement_path)
    display(settlement_df)
else:
    settlement_df = pd.DataFrame([{
        "contract_slug": contract_metadata["contract_slug"],
        "city": contract_metadata["city"],
        "settlement_date_local": contract_metadata["settlement_date_local"],
        "settlement_source": contract_metadata["settlement_source"],
        "settlement_variable": contract_metadata["settlement_variable"],
        "official_realised_temperature_c": np.nan,
        "retrieval_status": "settlement_file_not_found",
    }])
    display(settlement_df)

contract_metadata

,contract_slug,polymarket_event_url,city,settlement_date_local,settlement_source,settlement_variable,official_realised_temperature_c,station,unit,source_route,retrieval_status,source_reference,retrieval_time_utc
0,highest-temperature-in-hong-kong-on-may-30-2026,https://polymarket.com/event/highest-temperatu...,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,HKO,deg C,HKO open data CSV,matched_open_data_csv,https://data.weather.gov.hk/weatherAPI/opendat...,2026-06-16T23:22:26.556353+00:00


{'city': 'Hong Kong',
 'local_timezone': 'Asia/Hong_Kong',
 'contract_slug': 'highest-temperature-in-hong-kong-on-may-30-2026',
 'polymarket_event_url': 'https://polymarket.com/event/highest-temperature-in-hong-kong-on-may-30-2026',
 'settlement_date_local': '2026-05-30',
 'settlement_source': 'Hong Kong Observatory',
 'settlement_variable': 'Daily maximum temperature',
 'unit': 'deg C'}

## 3. Define local settlement window

For a daily maximum temperature contract, the target is not a single valid time. The target is the maximum temperature over a local calendar day.

For Hong Kong, the local day is converted to UTC. This makes the forecast and market timestamps comparable.

In [7]:
def local_day_window_utc(local_date_str, timezone_name):
    local_tz = ZoneInfo(timezone_name)

    local_start = datetime.fromisoformat(local_date_str).replace(tzinfo=local_tz)
    local_end = local_start + timedelta(days=1) - timedelta(seconds=1)
    local_midpoint = local_start + timedelta(hours=12)

    return {
        "target_local_date": local_date_str,
        "target_local_day_start": local_start,
        "target_local_day_end": local_end,
        "target_local_day_midpoint": local_midpoint,
        "target_local_day_start_utc": local_start.astimezone(timezone.utc),
        "target_local_day_end_utc": local_end.astimezone(timezone.utc),
        "target_local_day_midpoint_utc": local_midpoint.astimezone(timezone.utc),
    }

window = local_day_window_utc(
    contract_metadata["settlement_date_local"],
    contract_metadata["local_timezone"]
)

window_df = pd.DataFrame([{
    k: v.isoformat() if hasattr(v, "isoformat") else v
    for k, v in window.items()
}])

window_df

,target_local_date,target_local_day_start,target_local_day_end,target_local_day_midpoint,target_local_day_start_utc,target_local_day_end_utc,target_local_day_midpoint_utc
0,2026-05-30,2026-05-30T00:00:00+08:00,2026-05-30T23:59:59+08:00,2026-05-30T12:00:00+08:00,2026-05-29T16:00:00+00:00,2026-05-30T15:59:59+00:00,2026-05-30T04:00:00+00:00


## 4. Forecast timing examples

A daily maximum forecast may be constructed from several forecast valid times covering the local settlement day. However, each forecast is available only after its issue time.

This section creates a timing table for several example forecast runs. The market price should be sampled at the forecast issue time or at the first available market timestamp after the forecast becomes usable.

The `availability_lag_minutes` field allows later notebooks to account for the fact that a forecast may not be publicly downloadable exactly at the nominal run time.

In [9]:
def parse_utc(dt_str):
    return pd.Timestamp(dt_str, tz="UTC").to_pydatetime()


forecast_timing_rows = [
    {
        "forecast_model": "AIFS_or_ECMWF_example",
        "forecast_run_time_utc": parse_utc("2026-05-28 12:00:00"),
        "availability_lag_minutes": 0,
        "forecast_valid_time_utc": parse_utc("2026-05-30 12:00:00"),
        "timing_label": "two_day_valid_time_example",
    },
    {
        "forecast_model": "AIFS_or_ECMWF_example",
        "forecast_run_time_utc": parse_utc("2026-05-29 00:00:00"),
        "availability_lag_minutes": 0,
        "forecast_valid_time_utc": parse_utc("2026-05-30 12:00:00"),
        "timing_label": "one_and_half_day_valid_time_example",
    },
    {
        "forecast_model": "AIFS_or_ECMWF_example",
        "forecast_run_time_utc": parse_utc("2026-05-29 12:00:00"),
        "availability_lag_minutes": 0,
        "forecast_valid_time_utc": parse_utc("2026-05-30 12:00:00"),
        "timing_label": "one_day_valid_time_example",
    },
    {
        "forecast_model": "AIFS_or_ECMWF_example",
        "forecast_run_time_utc": parse_utc("2026-05-30 00:00:00"),
        "availability_lag_minutes": 0,
        "forecast_valid_time_utc": parse_utc("2026-05-30 12:00:00"),
        "timing_label": "same_day_valid_time_example",
    },
]

forecast_timing_df = pd.DataFrame(forecast_timing_rows)

# The issue time is the information time used for market comparison.
# If there is a publication/download delay, it should be added here.
forecast_timing_df["forecast_issue_time_utc"] = (
    forecast_timing_df["forecast_run_time_utc"]
    + pd.to_timedelta(forecast_timing_df["availability_lag_minutes"], unit="m")
)

# Settlement target: local daily maximum temperature.
forecast_timing_df["target_local_date"] = contract_metadata["settlement_date_local"]
forecast_timing_df["target_local_day_start_utc"] = window["target_local_day_start_utc"]
forecast_timing_df["target_local_day_end_utc"] = window["target_local_day_end_utc"]
forecast_timing_df["target_local_day_midpoint_utc"] = window["target_local_day_midpoint_utc"]

# Lead-time definitions.
forecast_timing_df["lead_time_to_valid_hours"] = (
    forecast_timing_df["forecast_valid_time_utc"]
    - forecast_timing_df["forecast_issue_time_utc"]
).dt.total_seconds() / 3600

forecast_timing_df["lead_time_to_day_start_hours"] = (
    forecast_timing_df["target_local_day_start_utc"]
    - forecast_timing_df["forecast_issue_time_utc"]
).dt.total_seconds() / 3600

forecast_timing_df["lead_time_to_day_midpoint_hours"] = (
    forecast_timing_df["target_local_day_midpoint_utc"]
    - forecast_timing_df["forecast_issue_time_utc"]
).dt.total_seconds() / 3600

forecast_timing_df["lead_time_to_day_end_hours"] = (
    forecast_timing_df["target_local_day_end_utc"]
    - forecast_timing_df["forecast_issue_time_utc"]
).dt.total_seconds() / 3600

# Whether the forecast is available before or during the local settlement day.
forecast_timing_df["forecast_issue_before_local_day_start"] = (
    forecast_timing_df["forecast_issue_time_utc"]
    <= forecast_timing_df["target_local_day_start_utc"]
)

forecast_timing_df["forecast_issue_before_local_day_midpoint"] = (
    forecast_timing_df["forecast_issue_time_utc"]
    <= forecast_timing_df["target_local_day_midpoint_utc"]
)

forecast_timing_df["forecast_issue_before_local_day_end"] = (
    forecast_timing_df["forecast_issue_time_utc"]
    <= forecast_timing_df["target_local_day_end_utc"]
)

forecast_timing_df["forecast_timing_category"] = np.select(
    [
        forecast_timing_df["forecast_issue_time_utc"]
        <= forecast_timing_df["target_local_day_start_utc"],

        forecast_timing_df["forecast_issue_time_utc"]
        <= forecast_timing_df["target_local_day_end_utc"],
    ],
    [
        "full_day_ex_ante",
        "within_day_update",
    ],
    default="after_target_day",
)

forecast_timing_df["market_price_timestamp_rule"] = (
    "first available Polymarket price at or after forecast_issue_time_utc"
)

display(forecast_timing_df)

,forecast_model,forecast_run_time_utc,availability_lag_minutes,forecast_valid_time_utc,timing_label,forecast_issue_time_utc,target_local_date,target_local_day_start_utc,target_local_day_end_utc,target_local_day_midpoint_utc,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,forecast_issue_before_local_day_start,forecast_issue_before_local_day_midpoint,forecast_issue_before_local_day_end,forecast_timing_category,market_price_timestamp_rule
0,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,0,2026-05-30 12:00:00+00:00,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,2026-05-30 04:00:00+00:00,48.0,28.0,40.0,51.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
1,AIFS_or_ECMWF_example,2026-05-29 00:00:00+00:00,0,2026-05-30 12:00:00+00:00,one_and_half_day_valid_time_example,2026-05-29 00:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,2026-05-30 04:00:00+00:00,36.0,16.0,28.0,39.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
2,AIFS_or_ECMWF_example,2026-05-29 12:00:00+00:00,0,2026-05-30 12:00:00+00:00,one_day_valid_time_example,2026-05-29 12:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,2026-05-30 04:00:00+00:00,24.0,4.0,16.0,27.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
3,AIFS_or_ECMWF_example,2026-05-30 00:00:00+00:00,0,2026-05-30 12:00:00+00:00,same_day_valid_time_example,2026-05-30 00:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,2026-05-30 04:00:00+00:00,12.0,-8.0,4.0,15.999722,False,True,True,within_day_update,first available Polymarket price at or after f...


## 5. Market timestamp rule

The safe comparison rule is:

`market_price_time_utc >= forecast_issue_time_utc`

This rule prevents the model from being compared with a later market price that may already contain additional information.

If a market price is not observed exactly at the issue time, the first available price after the issue time is used. The time gap is recorded.

In [11]:
def select_first_price_at_or_after(price_history, timestamp_col, price_col, issue_time):
    if price_history.empty:
        return {
            "market_price_time_utc": pd.NaT,
            "market_price": np.nan,
            "price_selection_status": "empty_price_history",
            "price_time_gap_minutes": np.nan,
        }

    df = price_history.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], utc=True)
    issue_ts = pd.Timestamp(issue_time).tz_convert("UTC") if pd.Timestamp(issue_time).tzinfo else pd.Timestamp(issue_time, tz="UTC")

    candidates = df[df[timestamp_col] >= issue_ts].sort_values(timestamp_col)

    if candidates.empty:
        return {
            "market_price_time_utc": pd.NaT,
            "market_price": np.nan,
            "price_selection_status": "no_price_after_issue_time",
            "price_time_gap_minutes": np.nan,
        }

    row = candidates.iloc[0]
    gap_minutes = (row[timestamp_col] - issue_ts).total_seconds() / 60

    return {
        "market_price_time_utc": row[timestamp_col],
        "market_price": row[price_col],
        "price_selection_status": "first_price_at_or_after_issue_time",
        "price_time_gap_minutes": gap_minutes,
    }

## 6. Retrieve Polymarket event metadata

This section retrieves the Hong Kong event metadata and identifies YES token IDs for the event's binary markets.

If the API structure changes or a market is unavailable, the notebook still preserves the timing framework above.

In [13]:
def safe_json_loads(value):
    if isinstance(value, (list, dict)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value)
    try:
        return json.loads(text)
    except Exception:
        try:
            return ast.literal_eval(text)
        except Exception:
            return None

def fetch_polymarket_event_by_slug(slug):
    urls = [
        f"https://gamma-api.polymarket.com/events/slug/{slug}",
        f"https://gamma-api.polymarket.com/events?slug={slug}",
    ]

    for url in urls:
        try:
            response = requests.get(url, timeout=30)
            print("URL:", url, "status:", response.status_code)
            if not response.ok:
                continue
            data = response.json()

            if isinstance(data, list):
                if len(data) > 0:
                    return data[0], url
            elif isinstance(data, dict):
                return data, url
        except Exception as e:
            print("Failed:", url, repr(e))

    return None, None

event_data, event_source_url = fetch_polymarket_event_by_slug(contract_metadata["contract_slug"])

if event_data is not None:
    (RAW_DIR / "polymarket_event_metadata.json").write_text(
        json.dumps(event_data, indent=2),
        encoding="utf-8"
    )
    print("Event title:", event_data.get("title") or event_data.get("question"))
    print("Event source:", event_source_url)
    print("Number of markets:", len(event_data.get("markets", [])))
else:
    print("No event metadata retrieved.")

URL: https://gamma-api.polymarket.com/events/slug/highest-temperature-in-hong-kong-on-may-30-2026 status: 200
Event title: Highest temperature in Hong Kong on May 30?
Event source: https://gamma-api.polymarket.com/events/slug/highest-temperature-in-hong-kong-on-may-30-2026
Number of markets: 11


In [14]:
def extract_yes_markets_from_event(event_data):
    if event_data is None:
        return pd.DataFrame()

    rows = []

    for market in event_data.get("markets", []):
        outcomes = safe_json_loads(market.get("outcomes"))
        outcome_prices = safe_json_loads(market.get("outcomePrices"))
        clob_token_ids = safe_json_loads(market.get("clobTokenIds"))

        if not isinstance(outcomes, list) or not isinstance(clob_token_ids, list):
            continue

        for i, outcome in enumerate(outcomes):
            if str(outcome).upper() != "YES":
                continue

            price = None
            if isinstance(outcome_prices, list) and i < len(outcome_prices):
                try:
                    price = float(outcome_prices[i])
                except Exception:
                    price = None

            rows.append({
                "event_slug": contract_metadata["contract_slug"],
                "market_id": market.get("id"),
                "condition_id": market.get("conditionId"),
                "question": market.get("question"),
                "market_slug": market.get("slug"),
                "outcome": outcome,
                "yes_token_id": clob_token_ids[i] if i < len(clob_token_ids) else None,
                "current_yes_price": price,
                "active": market.get("active"),
                "closed": market.get("closed"),
                "end_date": market.get("endDate"),
                "volume": market.get("volume"),
                "liquidity": market.get("liquidity"),
            })

    return pd.DataFrame(rows)

yes_markets_df = extract_yes_markets_from_event(event_data)
yes_markets_df

,event_slug,market_id,condition_id,question,market_slug,outcome,yes_token_id,current_yes_price,active,closed,end_date,volume,liquidity
0,highest-temperature-in-hong-kong-on-may-30-2026,2375825,0x5569d3884dea8ccc096111b9e89ca427ebb38ab4fe23...,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,2082456913986935254241388444978012953023591387...,0.0,True,True,2026-05-30T12:00:00Z,4275.984,0
1,highest-temperature-in-hong-kong-on-may-30-2026,2375826,0x6d149efe8b1c49a2954bac06f3b04cff52a8839f0193...,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,2943377179678179196313475284772204339317682077...,0.0,True,True,2026-05-30T12:00:00Z,1305.9940000000001,0
2,highest-temperature-in-hong-kong-on-may-30-2026,2375827,0x4203693c2820942f6b179bd2c8ffc6c2939f78d31210...,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,4617896828781800154535996881024729722157020410...,0.0,True,True,2026-05-30T12:00:00Z,441.104,0
3,highest-temperature-in-hong-kong-on-may-30-2026,2375828,0x77b763155d0bdbb1b8ddda14071efc9f1a2a549bab6c...,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,1040985512240410515356525715019651610521841919...,0.0,True,True,2026-05-30T12:00:00Z,1223.165774,0
4,highest-temperature-in-hong-kong-on-may-30-2026,2375829,0x1f5860fdc8410149a6617db677d2e759f98562368443...,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,8612028584163479635813164643298765384451615551...,0.0,True,True,2026-05-30T12:00:00Z,6673.330832000001,0
5,highest-temperature-in-hong-kong-on-may-30-2026,2375830,0x72e1d5919ff77bbeb52a8fd7112f11663a7c17b44ce3...,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,6762641211374974709079728339036642145194676044...,0.0,True,True,2026-05-30T12:00:00Z,11243.578412000004,None
6,highest-temperature-in-hong-kong-on-may-30-2026,2375831,0xd22b92ecaf607fc0bd789f16b865068619c900e464c6...,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,1264519449661180374335059763317437131424227980...,0.0,True,True,2026-05-30T12:00:00Z,24434.291205000012,None
7,highest-temperature-in-hong-kong-on-may-30-2026,2375832,0xb6554553818ce7e00edf18e1cd30d99d2092842ece00...,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,1030906326133351056965750446522265743053310917...,0.0,True,True,2026-05-30T12:00:00Z,13803.64386500001,None
8,highest-temperature-in-hong-kong-on-may-30-2026,2375833,0x799736b953678de34b1021e4adad3bb37e443f256090...,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,1866150743015618681650630480883665441327029368...,0.0,True,True,2026-05-30T12:00:00Z,34421.624074999985,None
9,highest-temperature-in-hong-kong-on-may-30-2026,2375834,0xc29d97aa6a1e68690aefd62542fe8892453d85bc8aef...,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-may-30-202...,Yes,3292528861173607221297409026900055602250951713...,1.0,True,True,2026-05-30T12:00:00Z,32671.39352099998,None


## 7. Retrieve YES price histories

The CLOB price-history endpoint is used to retrieve historical YES prices. The market comparison will use the first available price at or after the forecast issue time.

This is the key correction relative to a simple one-day-before-resolution comparison.

In [16]:
def fetch_price_history(token_id, interval="max"):
    if token_id is None or str(token_id) == "nan":
        return pd.DataFrame()

    url = "https://clob.polymarket.com/prices-history"
    params = {
        "market": str(token_id),
        "interval": interval,
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        if not response.ok:
            return pd.DataFrame({
                "yes_token_id": [token_id],
                "history_status": [f"http_{response.status_code}"],
                "history_error": [response.text[:300]],
            })

        data = response.json()
        history = data.get("history", data if isinstance(data, list) else [])

        rows = []
        for item in history:
            if not isinstance(item, dict):
                continue
            t = item.get("t") or item.get("timestamp")
            p = item.get("p") or item.get("price")
            rows.append({
                "yes_token_id": str(token_id),
                "timestamp_raw": t,
                "price": p,
                "history_status": "ok",
                "history_error": None,
            })

        hist = pd.DataFrame(rows)

        if hist.empty:
            return hist

        # Polymarket history timestamps are typically Unix seconds.
        hist["timestamp_utc"] = pd.to_datetime(hist["timestamp_raw"], unit="s", utc=True, errors="coerce")
        hist["price"] = pd.to_numeric(hist["price"], errors="coerce")
        hist = hist.dropna(subset=["timestamp_utc", "price"])
        hist = hist.sort_values("timestamp_utc")

        return hist

    except Exception as e:
        return pd.DataFrame({
            "yes_token_id": [token_id],
            "history_status": ["exception"],
            "history_error": [repr(e)],
        })

price_history_frames = []

for token_id in yes_markets_df["yes_token_id"].dropna().unique() if not yes_markets_df.empty else []:
    hist = fetch_price_history(token_id)
    if not hist.empty:
        price_history_frames.append(hist)

if price_history_frames:
    price_history_df = pd.concat(price_history_frames, ignore_index=True)
else:
    price_history_df = pd.DataFrame()

print("Price history rows:", len(price_history_df))
display(price_history_df.head())
display(price_history_df.tail())

Price history rows: 6727


,yes_token_id,timestamp_raw,price,history_status,history_error,timestamp_utc
0,2082456913986935254241388444978012953023591387...,1779942004,0.0050,ok,None,2026-05-28 04:20:04+00:00
1,2082456913986935254241388444978012953023591387...,1779942605,0.0050,ok,None,2026-05-28 04:30:05+00:00
2,2082456913986935254241388444978012953023591387...,1779943205,0.0025,ok,None,2026-05-28 04:40:05+00:00
3,2082456913986935254241388444978012953023591387...,1779943807,0.0025,ok,None,2026-05-28 04:50:07+00:00
4,2082456913986935254241388444978012953023591387...,1779944408,0.0025,ok,None,2026-05-28 05:00:08+00:00


,yes_token_id,timestamp_raw,price,history_status,history_error,timestamp_utc
6722,7673945377828659498594688521559031179409557337...,1780304405,0.0005,ok,None,2026-06-01 09:00:05+00:00
6723,7673945377828659498594688521559031179409557337...,1780305004,0.0005,ok,None,2026-06-01 09:10:04+00:00
6724,7673945377828659498594688521559031179409557337...,1780305605,0.0005,ok,None,2026-06-01 09:20:05+00:00
6725,7673945377828659498594688521559031179409557337...,1780306204,0.0005,ok,None,2026-06-01 09:30:04+00:00
6726,7673945377828659498594688521559031179409557337...,1780306805,0.0005,ok,None,2026-06-01 09:40:05+00:00


## 8. Align forecast issue times with market prices

For every forecast issue time and every YES market, the selected market price is the first price at or after the issue time.

In [18]:
alignment_rows = []

if yes_markets_df.empty:
    print("No YES markets available. Timing framework still remains valid.")
else:
    for _, timing_row in forecast_timing_df.iterrows():
        issue_time = timing_row["forecast_issue_time_utc"]

        for _, market_row in yes_markets_df.iterrows():
            token_id = str(market_row["yes_token_id"])
            token_hist = price_history_df[price_history_df["yes_token_id"].astype(str) == token_id].copy() if not price_history_df.empty else pd.DataFrame()

            selected = select_first_price_at_or_after(
                token_hist,
                timestamp_col="timestamp_utc",
                price_col="price",
                issue_time=issue_time,
            )

            alignment_rows.append({
                "contract_slug": contract_metadata["contract_slug"],
                "city": contract_metadata["city"],
                "target_local_date": contract_metadata["settlement_date_local"],
                "forecast_model": timing_row["forecast_model"],"timing_label": timing_row["timing_label"],
                "forecast_timing_category": timing_row["forecast_timing_category"],
                "forecast_issue_before_local_day_start": timing_row["forecast_issue_before_local_day_start"],
                "forecast_issue_before_local_day_midpoint": timing_row["forecast_issue_before_local_day_midpoint"],
                "forecast_issue_before_local_day_end": timing_row["forecast_issue_before_local_day_end"],
                "forecast_run_time_utc": timing_row["forecast_run_time_utc"],
                "forecast_issue_time_utc": timing_row["forecast_issue_time_utc"],
                "forecast_valid_time_utc": timing_row["forecast_valid_time_utc"],
                "target_local_day_start_utc": timing_row["target_local_day_start_utc"],
                "target_local_day_end_utc": timing_row["target_local_day_end_utc"],
                "lead_time_to_valid_hours": timing_row["lead_time_to_valid_hours"],
                "lead_time_to_day_start_hours": timing_row["lead_time_to_day_start_hours"],
                "lead_time_to_day_midpoint_hours": timing_row["lead_time_to_day_midpoint_hours"],
                "lead_time_to_day_end_hours": timing_row["lead_time_to_day_end_hours"],
                "market_id": market_row["market_id"],
                "market_slug": market_row["market_slug"],
                "question": market_row["question"],
                "yes_token_id": market_row["yes_token_id"],
                "market_price_time_utc": selected["market_price_time_utc"],
                "market_yes_price": selected["market_price"],
                "price_selection_status": selected["price_selection_status"],
                "price_time_gap_minutes": selected["price_time_gap_minutes"],
                "market_price_timestamp_rule": timing_row["market_price_timestamp_rule"],
            })

alignment_df = pd.DataFrame(alignment_rows)

if not settlement_df.empty:
    keep_cols = [
        c for c in [
            "contract_slug",
            "settlement_source",
            "settlement_variable",
            "official_realised_temperature_c",
            "retrieval_status",
        ]
        if c in settlement_df.columns
    ]

    settlement_small = settlement_df[keep_cols].drop_duplicates()
    if "contract_slug" in settlement_small.columns and not alignment_df.empty:
        alignment_df = alignment_df.merge(settlement_small, on="contract_slug", how="left")

alignment_df

,contract_slug,city,target_local_date,forecast_model,timing_label,forecast_timing_category,forecast_issue_before_local_day_start,forecast_issue_before_local_day_midpoint,forecast_issue_before_local_day_end,forecast_run_time_utc,forecast_issue_time_utc,forecast_valid_time_utc,target_local_day_start_utc,target_local_day_end_utc,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,market_id,market_slug,question,yes_token_id,market_price_time_utc,market_yes_price,price_selection_status,price_time_gap_minutes,market_price_timestamp_rule,settlement_source,settlement_variable,official_realised_temperature_c,retrieval_status
0,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375825,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,2082456913986935254241388444978012953023591387...,2026-05-28 12:00:07+00:00,0.0010,first_price_at_or_after_issue_time,0.116667,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,matched_open_data_csv
1,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375826,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,2943377179678179196313475284772204339317682077...,2026-05-28 12:00:12+00:00,0.0025,first_price_at_or_after_issue_time,0.200000,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,matched_open_data_csv
2,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375827,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,4617896828781800154535996881024729722157020410...,2026-05-28 12:00:12+00:00,0.0025,first_price_at_or_after_issue_time,0.200000,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,matched_open_data_csv
3,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375828,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,1040985512240410515356525715019651610521841919...,2026-05-28 12:00:04+00:00,0.0055,first_price_at_or_after_issue_time,0.066667,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,matched_open_data_csv
4,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375829,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,8612028584163479635813164643298765384451615551...,2026-05-28 12:00:04+00:00,0.0035,first_price_at_or_after_issue_time,0.066667,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,ma

## 9. Diagnostic checks

These checks confirm that the selected market price is never earlier than the forecast issue time.

In [20]:
if not alignment_df.empty:
    alignment_df["forecast_issue_time_utc"] = pd.to_datetime(alignment_df["forecast_issue_time_utc"], utc=True)
    alignment_df["market_price_time_utc"] = pd.to_datetime(alignment_df["market_price_time_utc"], utc=True, errors="coerce")

    valid_price_rows = alignment_df.dropna(subset=["market_price_time_utc"]).copy()

    if not valid_price_rows.empty:
        valid_price_rows["price_after_or_at_issue"] = (
            valid_price_rows["market_price_time_utc"] >= valid_price_rows["forecast_issue_time_utc"]
        )

        print("Rows with selected prices:", len(valid_price_rows))
        print("All selected prices at or after issue time:", valid_price_rows["price_after_or_at_issue"].all())
        display(valid_price_rows[[
            "timing_label",
            "forecast_issue_time_utc",
            "market_price_time_utc",
            "price_time_gap_minutes",
            "market_yes_price",
            "price_after_or_at_issue",
            "question",
        ]].head(30))
    else:
        print("No market price rows selected.")
else:
    print("Alignment table is empty.")

Rows with selected prices: 44
All selected prices at or after issue time: True


,timing_label,forecast_issue_time_utc,market_price_time_utc,price_time_gap_minutes,market_yes_price,price_after_or_at_issue,question
0,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:07+00:00,0.116667,0.0010,True,Will the highest temperature in Hong Kong be 2...
1,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:12+00:00,0.200000,0.0025,True,Will the highest temperature in Hong Kong be 2...
2,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:12+00:00,0.200000,0.0025,True,Will the highest temperature in Hong Kong be 2...
3,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:04+00:00,0.066667,0.0055,True,Will the highest temperature in Hong Kong be 2...
4,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:04+00:00,0.066667,0.0035,True,Will the highest temperature in Hong Kong be 2...
5,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:04+00:00,0.066667,0.0350,True,Will the highest temperature in Hong Kong be 2...
6,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:04+00:00,0.066667,0.0455,True,Will the highest temperature in Hong Kong be 2...
7,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:04+00:00,0.066667,0.1650,True,Will the highest temperature in Hong Kong be 3...
8,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:04+00:00,0.066667,0.3150,True,Will the highest temperature in Hong Kong be 3...
9,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:04+00:00,0.066667,0.2050,True,Will the highest temperature in Hong Kong be 3...


## 10. Save outputs

In [22]:
timing_output_path = PROCESSED_DIR / "hong_kong_forecast_timing_framework.csv"
alignment_output_path = PROCESSED_DIR / "hong_kong_market_price_time_alignment.csv"

forecast_timing_df.to_csv(timing_output_path, index=False)
alignment_df.to_csv(alignment_output_path, index=False)

print("Saved timing framework:", timing_output_path)
print("Saved market alignment table:", alignment_output_path)

display(forecast_timing_df)
display(alignment_df.head())

Saved timing framework: ../data/processed/time_alignment/hong_kong_forecast_timing_framework.csv
Saved market alignment table: ../data/processed/time_alignment/hong_kong_market_price_time_alignment.csv


,forecast_model,forecast_run_time_utc,availability_lag_minutes,forecast_valid_time_utc,timing_label,forecast_issue_time_utc,target_local_date,target_local_day_start_utc,target_local_day_end_utc,target_local_day_midpoint_utc,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,forecast_issue_before_local_day_start,forecast_issue_before_local_day_midpoint,forecast_issue_before_local_day_end,forecast_timing_category,market_price_timestamp_rule
0,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,0,2026-05-30 12:00:00+00:00,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,2026-05-30 04:00:00+00:00,48.0,28.0,40.0,51.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
1,AIFS_or_ECMWF_example,2026-05-29 00:00:00+00:00,0,2026-05-30 12:00:00+00:00,one_and_half_day_valid_time_example,2026-05-29 00:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,2026-05-30 04:00:00+00:00,36.0,16.0,28.0,39.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
2,AIFS_or_ECMWF_example,2026-05-29 12:00:00+00:00,0,2026-05-30 12:00:00+00:00,one_day_valid_time_example,2026-05-29 12:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,2026-05-30 04:00:00+00:00,24.0,4.0,16.0,27.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
3,AIFS_or_ECMWF_example,2026-05-30 00:00:00+00:00,0,2026-05-30 12:00:00+00:00,same_day_valid_time_example,2026-05-30 00:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,2026-05-30 04:00:00+00:00,12.0,-8.0,4.0,15.999722,False,True,True,within_day_update,first available Polymarket price at or after f...


,contract_slug,city,target_local_date,forecast_model,timing_label,forecast_timing_category,forecast_issue_before_local_day_start,forecast_issue_before_local_day_midpoint,forecast_issue_before_local_day_end,forecast_run_time_utc,forecast_issue_time_utc,forecast_valid_time_utc,target_local_day_start_utc,target_local_day_end_utc,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,market_id,market_slug,question,yes_token_id,market_price_time_utc,market_yes_price,price_selection_status,price_time_gap_minutes,market_price_timestamp_rule,settlement_source,settlement_variable,official_realised_temperature_c,retrieval_status
0,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375825,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,2082456913986935254241388444978012953023591387...,2026-05-28 12:00:07+00:00,0.0010,first_price_at_or_after_issue_time,0.116667,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,matched_open_data_csv
1,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375826,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,2943377179678179196313475284772204339317682077...,2026-05-28 12:00:12+00:00,0.0025,first_price_at_or_after_issue_time,0.200000,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,matched_open_data_csv
2,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375827,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,4617896828781800154535996881024729722157020410...,2026-05-28 12:00:12+00:00,0.0025,first_price_at_or_after_issue_time,0.200000,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,matched_open_data_csv
3,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375828,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,1040985512240410515356525715019651610521841919...,2026-05-28 12:00:04+00:00,0.0055,first_price_at_or_after_issue_time,0.066667,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,matched_open_data_csv
4,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,True,True,True,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375829,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,8612028584163479635813164643298765384451615551...,2026-05-28 12:00:04+00:00,0.0035,first_price_at_or_after_issue_time,0.066667,first available Polymarket price at or after f...,Hong Kong Observatory,Daily Maximum Temperature,32.6,ma

## 11. Interpretation

This notebook establishes the timing rule for later model comparison and trading analysis.

The forecast issue time represents the information time at which the model forecast becomes available. The Polymarket price used for comparison should be sampled at the forecast issue time, or at the first available market timestamp after that time. This prevents look ahead bias.

For a daily maximum temperature contract, the target is a local day rather than a single forecast valid time. Therefore, the notebook records the local day start, midpoint and end in UTC. It also records several lead time definitions, including lead time to the selected valid time and lead time to the local target day window.

The realised official temperature is used only after the event for scoring. It should not affect the selection of the market price timestamp.

The output table from this notebook provides the required timing structure for supervised threshold classification, probability comparison and later trading tests.

The distinction between full-day ex-ante forecasts and within-day updates is important. If the forecast issue time is before the local settlement day begins, the forecast is a full-day ex-ante forecast. If the forecast issue time is after the local day has started but before it ends, it is a within-day update. Within-day updates can still be analysed if the Polymarket price is sampled at the same information time, but they should be labelled separately because part of the local weather outcome window has already passed.